# DF-DPC — Optimizador con hardware fijo

## Restricciones de tu banco (fijas)

| Parámetro | Valor |
|---|---|
| Fuente | micro-focus, **40 kV** |
| L_max = d_sm + d_md | **1.2 m** |
| p_mask (período de máscara) | **200 µm** |
| p_det (pixel pitch) | **50 µm** |
| d_md mínima razonable | **≥ 20 cm** (propagación muestra → detector) |

## Variables / optimizables
- Ancho de slit $w$ (sistema de doble máscara)
- Tamaño focal $f_s$ (Hamamatsu L8121-03: 7 / 20 / 50 µm)
- Configuración (DPC-análoga / DF-DPC-análoga / arbitraria)
- $L = d_{sm}+d_{md}$ dentro de $[L_{min}, L_{max}]$
- Número de dithering steps

## ¿Por qué con `p_mask=200 µm, p_det=50 µm` no podemos usar N=2 ó N=3?

La condición de Moiré nulo del paper es
$$ p_{mask}\cdot M \;=\; N\cdot p_{det},\qquad N\in\mathbb{Z}^{+}, $$
y la magnificación geométrica de un sistema divergente cumple $M = (d_{sm}+d_{md})/d_{sm} \ge 1$.

Despejando $M$:
$$ M \;=\; \frac{N\,p_{det}}{p_{mask}} \;=\; N\cdot\frac{p_{det}}{p_{mask}}. $$

En **el paper** (p_mask=53, p_det=55) la razón es $\approx 1.04$, así que $N=2$ da $M\!\approx\!2$ y $N=3$ da $M\!\approx\!3$.

En **tu banco** (p_mask=200, p_det=50) la razón es $p_{det}/p_{mask} = 0.25$, así que
- $N=2 \Rightarrow M=0.5$  ⛔ imposible
- $N=3 \Rightarrow M=0.75$ ⛔ imposible (te daba $d_{md}<0$)
- $N=5 \Rightarrow M=1.25$ ✓ mínimo geométrico
- $N=8 \Rightarrow M=2.0$  ✓ **análogo a DPC del paper**
- $N=12 \Rightarrow M=3.0$ ✓ **análogo a DF-DPC del paper**

La calculadora barre $N$ y reporta cuáles son físicamente alcanzables con $L \le 1.2$ m y $d_{md}\ge 20$ cm.

In [ ]:
import warnings
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass, field
from typing import Optional, List, Tuple

%matplotlib inline
plt.rcParams['figure.dpi'] = 110

## 1. Parámetros

Los campos marcados como `FIJO` no se pueden cambiar en tu banco; el resto son optimizables.

In [ ]:
@dataclass
class Bench:
    # ===== FIJO =====
    p_mask_um: float = 200.0   # FIJO período de máscara
    p_det_um: float  = 50.0    # FIJO pixel pitch del detector
    L_max_m: float   = 1.20    # FIJO distancia fuente-detector máxima
    kV: float        = 40.0    # FIJO tubo a 40 kV (régimen no espectral del paper)

    # ===== VARIABLE / OPTIMIZABLE =====
    focal_spot_um: float = 7.0    # 7 / 20 / 50 µm (Hamamatsu L8121-03)
    L_m: Optional[float] = None   # punto de operación (None → se elige el óptimo)
    N: Optional[int]     = None   # nº pixels/período proyectado (None → se elige)
    slit_um: Optional[float] = None  # ancho de slit (None → se computa el óptimo)
    n_dither: int = 16            # pasos de dithering por período de máscara

    # ===== restricciones blandas (ajustables) =====
    d_md_min_m: float = 0.20      # propagación mínima muestra-detector (= d_md)
    L_min_m: float    = 0.30      # mínimo razonable de banco
    slit_min_um: float = 2.0      # límite mecánico inferior de la doble máscara
    slit_max_um: float = 150.0    # límite mecánico superior
    slit_safety: float = 0.85     # factor de seguridad sobre w_max teórico
    N_min: int = 2
    N_max: int = 30               # tope de búsqueda de N
    M_dev_tol: float = 0.02       # tolerancia para clasificar N como 'DPC-like'/'DF-DPC-like'

    # ===== detector (informativo) =====
    det_w_mm: float = 70.0
    det_h_mm: float = 14.0

bench = Bench()
bench

## 2. Relaciones básicas

$$ M(N) = N\,\frac{p_{det}}{p_{mask}}, \quad d_{sm} = \frac{L}{M}, \quad d_{md} = L\,\frac{M-1}{M} = L - d_{sm}. $$

Para que la geometría sea válida:
- $M \ge 1 + \varepsilon$ (i.e. $N \ge \lceil p_{mask}/p_{det}\rceil + 1$ si $p_{mask}/p_{det}$ es entero, o $\lceil p_{mask}/p_{det}\rceil$ si no)
- $d_{md}(L,M) \ge d_{md}^{min}$
- $L \le L_{max}$

De las dos últimas, dado $M$, el rango permitido de $L$ es
$$ L \in \Big[\;d_{md}^{min}\frac{M}{M-1},\;L_{max}\;\Big]. $$

In [ ]:
def M_of_N(b: Bench, N: int) -> float:
    return N * b.p_det_um / b.p_mask_um

def distances_from_L(L_m: float, M: float):
    d_sm = L_m / M
    return d_sm, L_m - d_sm

def L_min_for_M(b: Bench, M: float) -> float:
    """L mínimo (m) para que d_md ≥ d_md_min con esta M."""
    if M <= 1:
        return float('inf')
    return max(b.L_min_m, b.d_md_min_m * M / (M - 1))

def feasible_L_range(b: Bench, M: float):
    """Devuelve (L_lo, L_hi) en metros, o (nan,nan) si no hay rango."""
    lo = L_min_for_M(b, M)
    if lo > b.L_max_m:
        return float('nan'), float('nan')
    return lo, b.L_max_m

def w_max_for(b: Bench, M: float, fs_um: Optional[float] = None) -> float:
    """Ancho de slit máximo para que el beamlet quepa en 1 pixel del detector."""
    fs = fs_um if fs_um is not None else b.focal_spot_um
    penumbra = fs * (M - 1)
    if penumbra >= b.p_det_um:
        return float('nan')
    return (b.p_det_um - penumbra) / M

# sanity check: paper exacto
print("Sanity check con valores del paper (p_mask=53, p_det=55):")
paper = Bench(p_mask_um=53.0, p_det_um=55.0, L_max_m=1.80, d_md_min_m=0.05)
for N in (2, 3):
    M = M_of_N(paper, N)
    lo, hi = feasible_L_range(paper, M)
    print(f"  N={N}: M={M:.3f}  L feasible = [{lo*100:.1f}, {hi*100:.1f}] cm")

print("\nTu banco (p_mask=200, p_det=50):")
for N in (2, 3, 5, 8, 12):
    M = M_of_N(bench, N)
    if M <= 1:
        print(f"  N={N}: M={M:.3f}  ⛔ M ≤ 1 → imposible")
        continue
    lo, hi = feasible_L_range(bench, M)
    tag = '⛔' if np.isnan(lo) else 'OK'
    print(f"  N={N}: M={M:.3f}  L feasible = [{lo*100:.1f}, {hi*100:.1f}] cm  {tag}")

## 3. Barrido de N: qué configuraciones son físicamente viables

Para cada $N$ entero, reporta $M(N)$, el rango admitido de $L$, y, en el extremo $L=L_{max}$ (que maximiza $d_{md}$), el slit óptimo y los parámetros derivados.

In [ ]:
def scan_N(b: Bench, fs_um: Optional[float] = None) -> List[dict]:
    fs = fs_um if fs_um is not None else b.focal_spot_um
    rows = []
    for N in range(b.N_min, b.N_max + 1):
        M = M_of_N(b, N)
        if M <= 1 + 1e-6:
            rows.append(dict(N=N, M=M, feasible=False, reason='M ≤ 1 (geom. divergente)'))
            continue
        L_lo, L_hi = feasible_L_range(b, M)
        if np.isnan(L_lo):
            rows.append(dict(N=N, M=M, feasible=False,
                             reason=f'L_min requerida ({L_min_for_M(b,M)*100:.1f} cm) > L_max'))
            continue
        # punto de operación recomendado: L = L_hi (maximiza d_md → mejor propagación DF)
        L_op = L_hi
        d_sm, d_md = distances_from_L(L_op, M)
        w_max = w_max_for(b, M, fs)
        if np.isnan(w_max):
            w_opt = float('nan')
            slit_ok = False
        else:
            w_opt = np.clip(b.slit_safety * w_max, b.slit_min_um, min(b.slit_max_um, b.p_mask_um*0.9))
            slit_ok = (b.slit_min_um <= w_opt <= b.slit_max_um)
        rows.append(dict(
            N=N, M=M, feasible=True,
            L_lo_cm=L_lo*100, L_hi_cm=L_hi*100,
            L_op_cm=L_op*100, d_sm_cm=d_sm*100, d_md_cm=d_md*100,
            w_max_um=w_max, w_opt_um=w_opt, slit_ok=slit_ok,
            penumbra_det_um=fs*(M-1),
            px_eff_um=b.p_det_um/M,
            reason=''
        ))
    return rows

def print_scan(b: Bench, fs_um: Optional[float] = None):
    fs = fs_um if fs_um is not None else b.focal_spot_um
    print(f"Foco: {fs:.0f} µm   p_mask={b.p_mask_um:.0f} µm   p_det={b.p_det_um:.0f} µm   "
          f"L_max={b.L_max_m*100:.0f} cm   d_md_min={b.d_md_min_m*100:.0f} cm")
    print()
    rows = scan_N(b, fs)
    hdr = ['N','M','feas','L_lo[cm]','L_hi[cm]','d_sm[cm]','d_md[cm]','w_opt[µm]','penum[µm]','px_eff[µm]','tag']
    print('  '.join(h.ljust(10) for h in hdr))
    print('-'*120)
    targets = {2: 'DPC-like', 3: 'DF-DPC-like'}
    for r in rows:
        if not r['feasible']:
            tag = r['reason']
            print('  '.join([str(r['N']).ljust(10), f"{r['M']:.3f}".ljust(10), '⛔'.ljust(10),
                              '—'.ljust(10),'—'.ljust(10),'—'.ljust(10),'—'.ljust(10),
                              '—'.ljust(10),'—'.ljust(10),'—'.ljust(10), tag]))
        else:
            tag = ''
            for Mtarget, name in targets.items():
                if abs(r['M']-Mtarget) <= b.M_dev_tol*Mtarget:
                    tag = name
            w_str = f"{r['w_opt_um']:.2f}" if not np.isnan(r['w_opt_um']) else 'N/A'
            cells = [str(r['N']), f"{r['M']:.3f}", '✓',
                     f"{r['L_lo_cm']:.1f}", f"{r['L_hi_cm']:.1f}",
                     f"{r['d_sm_cm']:.1f}", f"{r['d_md_cm']:.1f}",
                     w_str, f"{r['penumbra_det_um']:.1f}",
                     f"{r['px_eff_um']:.2f}", tag]
            print('  '.join(c.ljust(10) for c in cells))

print_scan(bench)

## 4. Comparativa de focos disponibles

Para cada foco del Hamamatsu, qué N quedan abiertos y con qué slit óptimo.

In [ ]:
def compare_focal_spots(b: Bench, fs_list=(7.0, 20.0, 50.0)):
    print(f"{'foco[µm]':<10} {'N':<4} {'M':<6} {'d_md[cm]':<10} {'w_opt[µm]':<12} {'penum_det[µm]':<14}  status")
    print('-'*84)
    for fs in fs_list:
        rows = scan_N(b, fs)
        any_ok = False
        for r in rows:
            if not r['feasible']:
                continue
            # destacar solo los M=2 y M=3 (más cercanos a paper)
            if not (abs(r['M']-2)<=0.02 or abs(r['M']-3)<=0.02):
                continue
            any_ok = True
            status = 'OK' if (r['slit_ok'] and not np.isnan(r['w_opt_um'])) else 'slit fuera de bounds'
            if not np.isnan(r['w_opt_um']) and r['w_opt_um']*r['M']+r['penumbra_det_um'] > b.p_det_um:
                status = 'beamlet > p_det'
            w_str = f"{r['w_opt_um']:.2f}" if not np.isnan(r['w_opt_um']) else 'N/A'
            print(f"{fs:<10.0f} {r['N']:<4d} {r['M']:<6.2f} {r['d_md_cm']:<10.1f} {w_str:<12} {r['penumbra_det_um']:<14.2f}  {status}")
        if not any_ok:
            print(f"{fs:<10.0f} (ninguno de N=8/12 viable)")

compare_focal_spots(bench)

## 5. Optimizador global

Busca el mejor $(N, L, f_s, w)$ según una métrica simple — por defecto, **maximizar el ancho de slit** (proxy de señal), restringido a $M \in \{2, 3\}$ (paper). Cambiá `prefer_M` para forzar uno u otro.

In [ ]:
def optimize(b: Bench,
             prefer_M: float = 3.0,           # 2.0 = DPC-like, 3.0 = DF-DPC-like
             fs_choices=(7.0, 20.0, 50.0),
             metric: str = 'slit'              # 'slit' o 'd_md'
             ):
    """Devuelve (best, all_candidates)."""
    candidates = []
    warns = []
    for fs in fs_choices:
        for N in range(b.N_min, b.N_max + 1):
            M = M_of_N(b, N)
            if abs(M - prefer_M) > b.M_dev_tol * prefer_M:
                continue
            L_lo, L_hi = feasible_L_range(b, M)
            if np.isnan(L_lo):
                continue
            # Operate at L_hi (máxima propagación)
            L_op = L_hi
            d_sm, d_md = distances_from_L(L_op, M)
            w_max = w_max_for(b, M, fs)
            if np.isnan(w_max) or w_max < b.slit_min_um:
                continue
            w_opt = np.clip(b.slit_safety * w_max, b.slit_min_um, min(b.slit_max_um, b.p_mask_um*0.9))
            score = w_opt if metric == 'slit' else d_md
            candidates.append(dict(
                fs=fs, N=N, M=M, L_op_m=L_op,
                d_sm_cm=d_sm*100, d_md_cm=d_md*100,
                w_opt_um=w_opt, w_max_um=w_max,
                penumbra_det_um=fs*(M-1),
                illum_det_um=w_opt*M + fs*(M-1),
                px_eff_um=b.p_det_um/M,
                score=score,
            ))
    if not candidates:
        warns.append(f"[CRÍTICO] No hay configuración viable con M≈{prefer_M} en este banco.")
        for w in warns:
            warnings.warn(w); print(w)
        return None, []
    candidates.sort(key=lambda c: c['score'], reverse=True)
    best = candidates[0]
    print(f"=== Mejor configuración para M≈{prefer_M} (métrica: {metric}) ===")
    print(f"  Foco       : {best['fs']:.0f} µm")
    print(f"  N          : {best['N']}")
    print(f"  M          : {best['M']:.4f}")
    print(f"  L          : {best['L_op_m']*100:.1f} cm")
    print(f"  d_sm       : {best['d_sm_cm']:.1f} cm")
    print(f"  d_md       : {best['d_md_cm']:.1f} cm   (≥ {b.d_md_min_m*100:.0f} cm requerido)")
    print(f"  slit w_opt : {best['w_opt_um']:.2f} µm   (w_max teórico: {best['w_max_um']:.2f} µm, duty={best['w_opt_um']/b.p_mask_um:.3f})")
    print(f"  penumbra   : {best['penumbra_det_um']:.2f} µm en detector")
    print(f"  zona ilum. : {best['illum_det_um']:.2f} µm  (p_det = {b.p_det_um:.1f} µm)")
    print(f"  px efectivo: {best['px_eff_um']:.2f} µm en muestra")
    print()
    if len(candidates) > 1:
        print(f"Otros candidatos viables ({len(candidates)-1}):")
        for c in candidates[1:6]:
            print(f"  fs={c['fs']:>4.0f}µm  N={c['N']:>2d}  M={c['M']:.3f}  "
                  f"L={c['L_op_m']*100:5.1f}cm  d_md={c['d_md_cm']:5.1f}cm  "
                  f"w_opt={c['w_opt_um']:5.2f}µm")
    return best, candidates

_ = optimize(bench, prefer_M=3.0)

## 6. Punto de operación detallado

Resuelve un punto concreto con warnings completos. Si `N` o `L_m` o `slit_um` quedan en `None`, los completa automáticamente.

In [ ]:
def operating_point(b: Bench, prefer_M: Optional[float] = None) -> dict:
    warns = []
    # --- N ---
    if b.N is not None:
        N = b.N
    elif prefer_M is not None:
        # buscar N entero más cercano que satisfaga M = N·p_det/p_mask
        N = max(b.N_min, int(round(prefer_M * b.p_mask_um / b.p_det_um)))
    else:
        N = 12  # DF-DPC-like por default
    M = M_of_N(b, N)
    if M <= 1 + 1e-6:
        msg = f"[CRÍTICO] N={N} da M={M:.3f} ≤ 1 — geometría imposible."
        warnings.warn(msg); warns.append(msg)
        return dict(feasible=False, warnings=warns)

    # --- L ---
    L_lo, L_hi = feasible_L_range(b, M)
    if np.isnan(L_lo):
        msg = (f"[CRÍTICO] Con M={M:.3f} (N={N}) y d_md_min={b.d_md_min_m*100:.0f} cm "
               f"se necesita L ≥ {L_min_for_M(b,M)*100:.1f} cm pero L_max={b.L_max_m*100:.1f} cm.")
        warnings.warn(msg); warns.append(msg)
        return dict(feasible=False, N=N, M=M, warnings=warns)
    if b.L_m is None:
        L = L_hi   # default: maximizar propagación
    else:
        L = b.L_m
        if L < L_lo - 1e-9:
            msg = f"[CRÍTICO] L={L*100:.1f} cm < L_min({L_lo*100:.1f} cm) → d_md={(L*(M-1)/M)*100:.1f} cm < {b.d_md_min_m*100:.0f} cm."
            warnings.warn(msg); warns.append(msg)
        if L > L_hi + 1e-9:
            msg = f"[CRÍTICO] L={L*100:.1f} cm > L_max({L_hi*100:.1f} cm)."
            warnings.warn(msg); warns.append(msg)

    d_sm, d_md = distances_from_L(L, M)
    fs = b.focal_spot_um

    # --- Slit ---
    w_max = w_max_for(b, M, fs)
    if np.isnan(w_max):
        msg = (f"[CRÍTICO] Foco {fs:.1f} µm: penumbra ({fs*(M-1):.1f} µm) ≥ p_det ({b.p_det_um:.1f} µm) "
               f"para M={M:.3f}. Ningún slit > 0 funciona.")
        warnings.warn(msg); warns.append(msg)
        w = float('nan')
    else:
        w_opt = np.clip(b.slit_safety * w_max, b.slit_min_um, min(b.slit_max_um, b.p_mask_um*0.9))
        if b.slit_um is None:
            w = w_opt
            w_src = 'auto'
        else:
            w = b.slit_um
            w_src = 'user'
            if w > w_max:
                msg = (f"[ADVERTENCIA] slit usuario {w:.2f} µm > w_max teórico {w_max:.2f} µm: "
                       f"el beamlet excede 1 pixel — pérdida de contraste pixel-a-pixel.")
                warnings.warn(msg); warns.append(msg)
            if w > b.slit_max_um or w < b.slit_min_um:
                msg = f"[ADVERTENCIA] slit {w:.2f} µm fuera de bounds mecánicos [{b.slit_min_um},{b.slit_max_um}] µm."
                warnings.warn(msg); warns.append(msg)
        if w >= b.p_mask_um:
            msg = f"[CRÍTICO] slit {w:.2f} µm ≥ p_mask {b.p_mask_um:.0f} µm: la máscara queda transparente."
            warnings.warn(msg); warns.append(msg)

    # --- Dithering ---
    step_mask_um = b.p_mask_um / b.n_dither
    if not np.isnan(w) and step_mask_um > w:
        msg = (f"[ADVERTENCIA] paso de dithering ({step_mask_um:.2f} µm) > slit ({w:.2f} µm): "
               f"subí n_dither (actual {b.n_dither}) a ≥ {int(np.ceil(b.p_mask_um/w))} para muestrear el slit.")
        warnings.warn(msg); warns.append(msg)

    illum_det_um = (w * M + fs * (M - 1)) if not np.isnan(w) else float('nan')
    px_eff_um = b.p_det_um / M
    blur_det_um = fs * d_md / d_sm   # mismo que fs*(M-1) cuando muestra=máscara
    blur_obj_um = blur_det_um / M
    fov_w = b.det_w_mm / M
    fov_h = b.det_h_mm / M

    print(f"=== Punto de operación ===")
    print(f"  HW fijo : p_mask={b.p_mask_um:.0f} µm, p_det={b.p_det_um:.0f} µm, L_max={b.L_max_m*100:.0f} cm, kV={b.kV:.0f}")
    print(f"  Foco    : {fs:.0f} µm")
    print()
    print(f"  N       : {N}   (N·p_det = {N*b.p_det_um:.0f} µm proyectados / período)")
    print(f"  M       : {M:.4f}")
    print(f"  L       : {L*100:6.2f} cm   (rango admitido [{L_lo*100:.1f}, {L_hi*100:.1f}] cm)")
    print(f"  d_sm    : {d_sm*100:6.2f} cm")
    print(f"  d_md    : {d_md*100:6.2f} cm   (≥ {b.d_md_min_m*100:.0f} cm requerido)")
    print()
    if not np.isnan(w):
        print(f"  slit ({'auto' if b.slit_um is None else 'user'}) : {w:6.2f} µm   (w_max teórico: {w_max:.2f} µm, duty={w/b.p_mask_um:.3f})")
        print(f"  zona ilum. en det.: {illum_det_um:6.2f} µm  (p_det = {b.p_det_um:.0f} µm)")
    print(f"  penumbra (foco {fs:.0f} µm): {fs*(M-1):.2f} µm en detector, {fs*(M-1)/M:.2f} µm en muestra")
    print(f"  pixel efectivo en muestra: {px_eff_um:.2f} µm/px")
    print(f"  FOV en muestra : {fov_w:.1f} × {fov_h:.1f} mm")
    print()
    print(f"  Dithering ({b.n_dither} pasos/período):")
    print(f"     paso en máscara/muestra : {step_mask_um:.2f} µm")
    print(f"     paso equivalente en det.: {step_mask_um*M:.2f} µm")
    if warns:
        print()
        print("  ⚠ Warnings:")
        for w_msg in warns:
            print(f"    - {w_msg}")
    return dict(feasible=True, N=N, M=M, L_m=L, d_sm_m=d_sm, d_md_m=d_md,
                slit_um=w, illum_det_um=illum_det_um, px_eff_um=px_eff_um,
                fov_mm=(fov_w, fov_h), warnings=warns)

_ = operating_point(bench, prefer_M=3.0)

## 7. Visualización: rango admisible (L, d_md) por configuración

In [ ]:
def plot_feasibility_map(b: Bench, N_list=(5, 8, 10, 12, 16)):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.3))
    # (a) d_md vs L para varios N
    ax = axes[0]
    L = np.linspace(0.05, b.L_max_m, 200)
    for N in N_list:
        M = M_of_N(b, N)
        if M <= 1: continue
        d_md = L * (M - 1) / M
        ax.plot(L*100, d_md*100, label=f'N={N}  M={M:.2f}')
    ax.axhline(b.d_md_min_m*100, color='red', ls='--', label=f'd_md_min={b.d_md_min_m*100:.0f} cm')
    ax.axvline(b.L_max_m*100, color='black', ls=':', label=f'L_max={b.L_max_m*100:.0f} cm')
    ax.set_xlabel('L = d_sm + d_md  [cm]'); ax.set_ylabel('d_md  [cm]')
    ax.set_title('Propagación d_md vs L'); ax.grid(alpha=.3); ax.legend(fontsize=7)

    # (b) ventana de L admisible para cada N
    ax = axes[1]
    Ns = list(range(b.N_min, b.N_max + 1))
    los, his = [], []
    for N in Ns:
        M = M_of_N(b, N)
        if M <= 1:
            los.append(np.nan); his.append(np.nan); continue
        lo, hi = feasible_L_range(b, M)
        los.append(lo*100 if not np.isnan(lo) else np.nan)
        his.append(hi*100 if not np.isnan(hi) else np.nan)
    los = np.array(los); his = np.array(his)
    ok = ~np.isnan(los)
    ax.bar(np.array(Ns)[ok], (his-los)[ok], bottom=los[ok], width=0.7, color='tab:blue', alpha=.6)
    for N, M_val in [(8, 2.0), (12, 3.0)]:
        if N in Ns:
            ax.axvline(N, color='tab:orange', ls=':', alpha=.7)
    ax.axhline(b.L_max_m*100, color='black', ls=':')
    ax.set_xlabel('N (pixels / período proyectado)')
    ax.set_ylabel('rango admitido de L  [cm]')
    ax.set_title('Ventanas de L admisibles por N')
    ax.grid(alpha=.3)
    fig.suptitle(f"p_mask={b.p_mask_um:.0f} µm  p_det={b.p_det_um:.0f} µm  L_max={b.L_max_m*100:.0f} cm  d_md_min={b.d_md_min_m*100:.0f} cm")
    fig.tight_layout()
    return fig

plot_feasibility_map(bench);

In [ ]:
def plot_slit_window(b: Bench, fs_list=(7.0, 20.0, 50.0), N_focus=(8, 12)):
    fig, axes = plt.subplots(1, len(N_focus), figsize=(6*len(N_focus), 4.2))
    if len(N_focus) == 1: axes = [axes]
    for ax, N in zip(axes, N_focus):
        M = M_of_N(b, N)
        if M <= 1:
            ax.set_title(f'N={N}  M={M:.2f}  (imposible)'); ax.axis('off'); continue
        w_range = np.linspace(0.5, min(b.slit_max_um, b.p_mask_um*0.95), 200)
        for fs in fs_list:
            zone = w_range * M + fs * (M - 1)
            line, = ax.plot(w_range, zone, label=f'foco {fs:.0f} µm')
            if fs * (M-1) < b.p_det_um:
                w_max = (b.p_det_um - fs*(M-1)) / M
                ax.axvline(w_max, color=line.get_color(), ls=':', alpha=0.5)
        ax.axhline(b.p_det_um, color='red', ls='--', label=f'p_det={b.p_det_um:.0f} µm')
        ax.axvspan(b.slit_min_um, b.slit_max_um, color='gray', alpha=0.08,
                   label=f'slit alcanzable [{b.slit_min_um:.0f},{b.slit_max_um:.0f}]')
        ax.set_xlabel('ancho de slit w [µm]'); ax.set_ylabel('zona iluminada en det [µm]')
        ax.set_title(f'N={N}   M={M:.3f}')
        ax.grid(alpha=.3); ax.legend(fontsize=7, loc='upper left')
    fig.suptitle('Beamlet en detector vs ancho de slit (debe caber en 1 pixel)')
    fig.tight_layout()
    return fig

plot_slit_window(bench);

## 8. Diagrama de la geometría

In [ ]:
def plot_geometry(b: Bench, op: dict):
    if not op.get('feasible'):
        print('Operating point no factible — no se dibuja diagrama.'); return None
    L_m = op['L_m']; d_sm = op['d_sm_m']; d_md = op['d_md_m']; M = op['M']
    w_um = op['slit_um']
    fig, ax = plt.subplots(figsize=(11, 3.2))
    y0 = 0
    ax.hlines(y0, 0, L_m*100, color='lightgray', lw=1)
    ax.plot(0, y0, 'o', color='gold', markersize=14, markeredgecolor='k')
    ax.text(0, y0+0.18, f"Fuente\n({b.focal_spot_um:.0f} µm, {b.kV:.0f} kV)", ha='center', fontsize=8)
    xm = d_sm*100
    ax.vlines(xm, y0-0.12, y0+0.12, color='tab:orange', lw=4)
    slit_lbl = f"{w_um:.1f} µm" if not np.isnan(w_um) else 'N/A'
    ax.text(xm, y0+0.18, f"Máscara\np={b.p_mask_um:.0f} µm\nslit={slit_lbl}", ha='center', fontsize=8)
    ax.vlines(xm+0.4, y0-0.08, y0+0.08, color='tab:green', lw=3)
    ax.text(xm+0.4, y0-0.28, 'Muestra', ha='center', fontsize=8, color='tab:green')
    xd = (d_sm+d_md)*100
    ax.vlines(xd, y0-0.15, y0+0.15, color='tab:blue', lw=5)
    ax.text(xd, y0+0.21, f"Detector\np={b.p_det_um:.0f} µm", ha='center', fontsize=8)
    ax.annotate('', xy=(xm, y0-0.35), xytext=(0, y0-0.35), arrowprops=dict(arrowstyle='<->'))
    ax.text(xm/2, y0-0.42, f"d_sm = {d_sm*100:.1f} cm", ha='center', fontsize=9)
    ax.annotate('', xy=(xd, y0-0.35), xytext=(xm, y0-0.35), arrowprops=dict(arrowstyle='<->'))
    ax.text((xm+xd)/2, y0-0.42, f"d_md = {d_md*100:.1f} cm", ha='center', fontsize=9)
    ax.annotate('', xy=(xd, y0+0.55), xytext=(0, y0+0.55), arrowprops=dict(arrowstyle='<->'))
    ax.text(xd/2, y0+0.6, f"L = {L_m*100:.1f} cm   (M = {M:.3f},  N = {op['N']})", ha='center', fontsize=9, weight='bold')
    ax.set_ylim(-0.6, 0.85); ax.set_xlim(-3, L_m*100+3)
    ax.set_yticks([]); ax.set_xlabel('posición a lo largo del eje óptico [cm]')
    ax.set_title(f"Geometría — banco fijo (DF-DPC-análoga si M≈3)")
    fig.tight_layout()
    return fig

op = operating_point(bench, prefer_M=3.0)
plot_geometry(bench, op);

## 9. Test rápido

Editá la celda con tus valores. Las celdas FIJAS no se tocan — el resto es libre.

In [ ]:
mi_bench = Bench(
    # FIJO --- no tocar ---
    p_mask_um = 200.0,
    p_det_um  = 50.0,
    L_max_m   = 1.20,
    kV        = 40.0,
    # variable ---
    focal_spot_um = 7.0,
    L_m       = None,     # None → usa L_max
    N         = None,     # None → se elige según prefer_M
    slit_um   = None,     # None → óptimo
    n_dither  = 16,
    # restricciones blandas ---
    d_md_min_m = 0.20,
    L_min_m    = 0.30,
    slit_min_um = 2.0,
    slit_max_um = 150.0,
    slit_safety = 0.85,
    N_max = 30,
)

print("### Scan de N ###\n"); print_scan(mi_bench); print()
print("### Comparativa de focos ###\n"); compare_focal_spots(mi_bench); print()
print("### Optimizador (M=3, DF-DPC-like) ###\n"); best, _ = optimize(mi_bench, prefer_M=3.0); print()
print("### Punto de operación ###\n"); op = operating_point(mi_bench, prefer_M=3.0)
plot_feasibility_map(mi_bench)
plot_slit_window(mi_bench)
plot_geometry(mi_bench, op);

### Stress test 1 — reproducir el bug original
Si forzamos N=3 con tu hardware (`p_mask=200, p_det=50`) debe emitir warning crítico (no warning silencioso d_md negativo).

In [ ]:
bad = Bench(p_mask_um=200, p_det_um=50, L_max_m=1.2, N=3, focal_spot_um=20.0)
_ = operating_point(bad)

### Stress test 2 — DPC-análoga (M=2)

In [ ]:
_ = optimize(mi_bench, prefer_M=2.0)
print()
op2 = operating_point(Bench(p_mask_um=200, p_det_um=50, L_max_m=1.2,
                            focal_spot_um=20.0, N=8))
plot_geometry(Bench(p_mask_um=200, p_det_um=50, L_max_m=1.2,
                    focal_spot_um=20.0, N=8), op2);